# 04 · The Tool Layer — How LLM Agents Use the Stack

Every scientific capability is exposed as a `BaseTool` subclass with a
Pydantic input schema. This is exactly what the MCP server serializes
for LLM agents to consume — and what the LangGraph workflow nodes call
internally.

This notebook walks through the agent-facing API directly, so you can
see what your agents will actually see.

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from metaopticsai.store.memory import InMemoryArtifactStore
from metaopticsai.physics.backends.base import BackendRegistry
from metaopticsai.physics.backends.analytical import AnalyticalBackend
from metaopticsai.tools import build_tool_registry

store = InMemoryArtifactStore()
backends = BackendRegistry(); backends.register(AnalyticalBackend())
tools = build_tool_registry(store, backends)

## 1 · Inspect the registry

The registry advertises *name → description* — the same surface an LLM
sees as tool specs.

In [2]:
for name in sorted(tools.names()):
    tool = tools.get(name)
    print(f"· {name}")
    print(f"    {tool.description[:120]}{'...' if len(tool.description) > 120 else ''}")

· analyze_mtf
    Compute the radially-averaged MTF and the diffraction-limit MTF from a stored PSF. Returns handle plus MTF50 (lp/mm).
· analyze_psf
    Compute the point-spread function for a stored phase mask. Returns a handle to the PSF plus Strehl ratio and FWHM.
· compute_center_intensity
    On-axis intensity at PSF center — useful as a differentiable FOM.
· compute_max_intensity
    Maximum intensity of a stored PSF / propagated field. Higher = better-focused.
· compute_mtf_volume
    Volume under the normalized MTF surface of a stored PSF. Used internally by gradient-based Strehl optimization.
· export_gds
    Export a metalens design as a fabrication-ready GDSII file.
· generate_phase_mask
    Generate a 2-D continuous phase mask for one of three canonical elements: fzl (Fresnel zone lens), axicon (Bessel beam),...
· get_fdtd_library
    List the wavelengths, materials, and unit-cell shapes available in the built-in FDTD reference library.
· get_material_index
    Look up com

## 2 · Inspect a tool's Input schema

Every tool's `.Input` is a Pydantic class. JSON-schema export drives the
MCP Inspector UI and Claude's tool-spec generation.

In [3]:
sweep_tool = tools.get("run_rcwa_sweep")
schema = sweep_tool.Input.model_json_schema()
print(json.dumps(schema, indent=2)[:1200])

{
  "properties": {
    "wavelength_nm": {
      "description": "Design wavelength in nm, e.g. 532 for green.",
      "exclusiveMaximum": 20000,
      "exclusiveMinimum": 200,
      "title": "Wavelength Nm",
      "type": "number"
    },
    "pillar_material": {
      "default": "TiO2",
      "description": "Meta-atom material. TiO2=visible, Si=NIR, GaN=UV/blue, SiN=CMOS visible.",
      "enum": [
        "TiO2",
        "Si",
        "GaN",
        "SiN"
      ],
      "title": "Pillar Material",
      "type": "string"
    },
    "pillar_height_nm": {
      "default": 600.0,
      "description": "Pillar height in nm. Increase to widen phase coverage.",
      "exclusiveMaximum": 3000,
      "exclusiveMinimum": 50,
      "title": "Pillar Height Nm",
      "type": "number"
    },
    "period_nm": {
      "default": 350.0,
      "description": "Square unit-cell period. 0.6\u20131.1\u00d7 wavelength is the safe regime.",
      "exclusiveMaximum": 2000,
      "exclusiveMinimum": 100,
      

## 3 · Call a tool the agent way

Agents pass a JSON dict. The tool validates against `Input`, dispatches
`_run`, and validates the output against `Output`. Bad input → clear
Pydantic error → the agent retries.

In [4]:
sweep_result = tools.get("run_rcwa_sweep").call_sync({
    "wavelength_nm": 532,
    "pillar_material": "TiO2",
    "pillar_height_nm": 600.0,
    "period_nm": 350.0,
    "min_diameter_nm": 50.0,
    "max_diameter_nm": 250.0,
    "n_samples": 30,
    "xy_harmonics": 5,
})
# Note: the LLM only ever sees this lightweight summary, never the array.
print(json.dumps(sweep_result, indent=2, default=str))

{
  "handle": "21156443cdb2",
  "phase_coverage_2pi_fraction": 0.7016461739372571,
  "mean_transmission": 0.994909090909091,
  "elapsed_s": 0.0,
  "n_samples": 30,
  "backend": "analytical",
  "advice": "Phase coverage < 0.9 \u2014 increase pillar_height_nm or widen the diameter range."
}


In [5]:
# Show what a deliberately invalid call looks like — the kind of error
# the agent has to recover from.
try:
    tools.get("run_rcwa_sweep").call_sync({
        "wavelength_nm": 532,
        "pillar_material": "Unobtainium",   # not in the materials list
        "min_diameter_nm": 50, "max_diameter_nm": 40,  # min > max
    })
except Exception as e:
    print(type(e).__name__, "→")
    print(str(e)[:600])

ValidationError →
1 validation error for RCWASweepInput
pillar_material
  Input should be 'TiO2', 'Si', 'GaN' or 'SiN' [type=literal_error, input_value='Unobtainium', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error


## 4 · Multi-step orchestration the way an agent would do it

Below: an LLM-style sequence — sweep, mask, PSF, MTF — chaining handles
through. The agent never touches arrays; it just routes handles.

In [6]:
# Step 1 — sweep already done above; reuse sweep_result["handle"]
# Step 2 — generate the phase mask
mask_result = tools.get("generate_phase_mask").call_sync({
    "mask_type": "fzl",
    "wavelength_nm": 532,
    "focal_length_um": 100.0,
    "diameter_um": 50.0,
    "pixel_size_um": 0.35,
})
# Step 3 — PSF analysis
psf_result = tools.get("analyze_psf").call_sync({
    "mask_handle": mask_result["handle"],
    "wavelength_nm": 532,
    "pixel_size_um": 0.35,
    "focal_length_um": 100.0,
    "pad_factor": 4,
})
# Step 4 — MTF analysis
mtf_result = tools.get("analyze_mtf").call_sync({
    "psf_handle": psf_result["handle"],
    "pixel_size_um": 0.35,
    "wavelength_nm": 532,
    "focal_length_um": 100.0,
})

print("Sequence summary (this is what the LLM accumulates as context):\n")
print(json.dumps({
    "sweep": sweep_result,
    "mask": mask_result,
    "psf": psf_result,
    "mtf": mtf_result,
}, indent=2, default=str))

Sequence summary (this is what the LLM accumulates as context):

{
  "sweep": {
    "handle": "21156443cdb2",
    "phase_coverage_2pi_fraction": 0.7016461739372571,
    "mean_transmission": 0.994909090909091,
    "elapsed_s": 0.0,
    "n_samples": 30,
    "backend": "analytical",
    "advice": "Phase coverage < 0.9 \u2014 increase pillar_height_nm or widen the diameter range."
  },
  "mask": {
    "handle": "122bd2b2b136",
    "shape": [
      142,
      142
    ],
    "phase_min_rad": 0.0,
    "phase_max_rad": 6.2823438436377534,
    "mask_type": "fzl"
  },
  "psf": {
    "handle": "923d0b456f2c",
    "strehl_ratio": 0.0012921796054710516,
    "fwhm_x_um": 2.6808,
    "shape": [
      568,
      568
    ],
    "wavelength_nm": 532.0
  },
  "mtf": {
    "handle": "492e865b4742",
    "mtf50_lpmm": 10.44,
    "cutoff_lpmm": 1418.5110663983905
  }
}


## 5 · State recovery via `list_artifacts`

Agents lose context across long conversations. `list_artifacts` is the
"where am I?" tool — it returns every handle currently in the store so
the agent can pick up where it left off.

In [7]:
result = tools.get("list_artifacts").call_sync({})
for art in result:
    print(f"  {art['kind']:<16} {art['handle']}  metadata: {art['metadata']}")

  mtf              492e865b4742  metadata: {}
  psf              923d0b456f2c  metadata: {'wavelength_nm': 532.0}
  phase_mask       122bd2b2b136  metadata: {'mask_type': 'fzl', 'wavelength_nm': 532.0, 'diameter_um': 50.0, 'pixel_size_um': 0.35}
  rcwa_sweep       21156443cdb2  metadata: {'wavelength_nm': 532.0, 'material': 'TiO2', 'backend': 'analytical'}


That handles-as-references discipline is the backbone of everything
else: it's why the same tools work locally, in pytest, in LangGraph
nodes, and over an MCP-stdio connection from Claude Desktop.